# LegalEagle CUAD Preprocessing

Prepare focused CUAD labels for LegalBERT fine-tuning. Upload the CUAD JSON or CSV export to Colab, normalize clause labels, and export train/validation/test splits.

In [ ]:
FOCUSED_LABELS = [
    'termination', 'assignment', 'indemnification', 'limitation of liability',
    'confidentiality', 'exclusivity', 'non-compete / non-solicit',
    'governing law', 'change of control', 'most favored nation',
    'audit rights', 'revenue / payment obligations'
]
LABEL_ALIASES = {
    'termination': ['termination'],
    'assignment': ['assignment'],
    'indemnification': ['indemnification', 'indemnity'],
    'limitation of liability': ['limitation of liability', 'liability cap'],
    'confidentiality': ['confidentiality'],
    'exclusivity': ['exclusivity'],
    'non-compete / non-solicit': ['non-compete', 'non-solicit', 'non compete', 'non solicit'],
    'governing law': ['governing law'],
    'change of control': ['change of control'],
    'most favored nation': ['most favored nation', 'mfn'],
    'audit rights': ['audit rights'],
    'revenue / payment obligations': ['revenue', 'payment', 'fees']
}


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Expected columns after adapting CUAD export: contract_id, text, label
raw_path = '/content/cuad_clauses.csv'
df = pd.read_csv(raw_path)
df['label_norm'] = df['label'].str.lower().str.strip()

def map_label(label):
    for canonical, aliases in LABEL_ALIASES.items():
        if any(alias in label for alias in aliases):
            return canonical
    return None

df['focused_label'] = df['label_norm'].map(map_label)
focused = df.dropna(subset=['focused_label']).dropna(subset=['text'])
focused = focused[['contract_id', 'text', 'focused_label']].rename(columns={'focused_label': 'label'})
focused.head()


In [ ]:
train_df, temp_df = train_test_split(focused, test_size=0.3, random_state=42, stratify=focused['label'])
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_df.to_json('/content/legaleagle_train.jsonl', orient='records', lines=True)
valid_df.to_json('/content/legaleagle_valid.jsonl', orient='records', lines=True)
test_df.to_json('/content/legaleagle_test.jsonl', orient='records', lines=True)
print(len(train_df), len(valid_df), len(test_df))
